# Challenge: Backtest on Other Datasets

## Download data from `yfinance`

In [134]:
import yfinance as yf

In [135]:
ticker = 'ETH-USD'
df_ETHUSD= yf.download(ticker, multi_level_index=False, auto_adjust=False)
df_ETHUSD

[*********************100%***********************]  1 of 1 completed


,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2026-04-27,2303.063965,2303.063965,2403.726562,2267.409424,2369.841797,17879618684
2026-04-28,2289.419678,2289.419678,2310.442139,2259.057373,2303.079834,12761522276
2026-04-29,2253.415771,2253.415771,2345.939697,2221.223877,2289.412109,21884731493
2026-04-30,2256.251221,2256.251221,2277.718262,2232.130859,2253.484375,12309496389
2026-05-01,2295.093750,2295.093750,2324.788086,2256.078125,2256.344971,13243880777
2026-05-02,2316.231934,2316.231934,2339.743652,2292.722656,2295.044434,6905015549
2026-05-03,2321.635742,2321.635742,2354.710205,2297.553223,2316.210938,9861070993
2026-05-04,2346.396973,2346.396973,2397.608398,2309.260986,2321.816406,25497505008
2026-05-05,2361.182617,2361.182617,2398.833496,2344.934326,2346.415039,17739612383


## Preprocess the data

### Filter the date range

- Since 1 year ago at least

In [136]:
df_ETHUSD_yahoo = df_ETHUSD.loc['2024-05-26':,:].copy()

### Create the target variable

#### Percentage change

- Percentage change on `Adj Close` for tomorrow

In [137]:
df_ETHUSD_yahoo['change_tomorrow']= df_ETHUSD_yahoo.Close.pct_change(-1)* 100 * -1

#### Drop rows with any missing data

In [138]:
df_ETHUSD_yahoo= df_ETHUSD_yahoo.dropna().copy()

#### Change sign

Did the stock go up or down?

In [139]:
import numpy as np

In [140]:
df_ETHUSD_yahoo['change_tomorrow_direction']= np.where(df_ETHUSD_yahoo.change_tomorrow > 0 ,'UP','DOWN')

In [141]:
df_ETHUSD_yahoo

,Adj Close,Close,High,Low,Open,Volume,change_tomorrow,change_tomorrow_direction
Date,,,,,,,,
2026-04-27,2303.063965,2303.063965,2403.726562,2267.409424,2369.841797,17879618684,-0.595971,DOWN
2026-04-28,2289.419678,2289.419678,2310.442139,2259.057373,2303.079834,12761522276,-1.597748,DOWN
2026-04-29,2253.415771,2253.415771,2345.939697,2221.223877,2289.412109,21884731493,0.125671,UP
2026-04-30,2256.251221,2256.251221,2277.718262,2232.130859,2253.484375,12309496389,1.692416,UP
2026-05-01,2295.093750,2295.093750,2324.788086,2256.078125,2256.344971,13243880777,0.912611,UP
2026-05-02,2316.231934,2316.231934,2339.743652,2292.722656,2295.044434,6905015549,0.232759,UP
2026-05-03,2321.635742,2321.635742,2354.710205,2297.553223,2316.210938,9861070993,1.055287,UP
2026-05-04,2346.396973,2346.396973,2397.608398,2309.260986,2321.816406,25497505008,0.626197,UP
2026-05-05,2361.182617,2361.182617,2398.833496,2344.934326,2346.415039,17739612383,-0.441578,DOWN


## Compute Machine Learning model

Proposal: Random Forest within `ensemble` module of `sklearn` library

In [142]:
from sklearn.ensemble import RandomForestClassifier

In [143]:
model = RandomForestClassifier(max_depth=7, random_state=42)

In [144]:
y = df_ETHUSD_yahoo.change_tomorrow_direction
X = df_ETHUSD_yahoo.drop(columns=['change_tomorrow', 'change_tomorrow_direction'])

In [145]:
target = df_ETHUSD_yahoo.change_tomorrow_direction

In [146]:
df_explanatory = df_ETHUSD_yahoo.drop(columns=['change_tomorrow', 'change_tomorrow_direction'])

In [147]:
model.fit(X, y)

RandomForestClassifier(max_depth=7, random_state=42)

## Backtesting

### Create the Strategy

In [148]:
from backtesting import Backtest, Strategy

In [149]:
explanatory_today= df_explanatory.iloc[[-1],:]

### Run the Backtest

In [150]:
import pandas as pd 

In [153]:
class SimpleClassificationUD(Strategy):
    def init(self):
        self.model = model
        self.already_bought = False

    def next(self):
        explanatory_today = self.data.df.iloc[-1:, :]
        forecast_tomorrow = self.model.predict(explanatory_today)[0]
        
        # conditions to sell or buy
        if forecast_tomorrow == 1 and self.already_bought == False:
            self.buy()
            self.already_bought = True
        elif forecast_tomorrow == -1 and self.already_bought == True:
            self.sell()
            self.already_bought = False
        else:
            pass

In [154]:
bt = Backtest(
    X, SimpleClassificationUD, cash=10000,
    commission=.002, exclusive_orders=True
)

### Show the report in a DataFrame

In [155]:
bt.run()

stats = bt.run()
pd.DataFrame(stats).reset_index()

,index,0
0,Start,2026-04-27 00:00:00
1,End,2026-05-26 00:00:00
2,Duration,29 days 00:00:00
3,Exposure Time [%],0.0
4,Equity Final [$],10000.0
5,Equity Peak [$],10000.0
6,Return [%],0.0
7,Buy & Hold Return [%],-10.082473
8,Return (Ann.) [%],0.0
9,Volatility (Ann.) [%],0.0


## Plot the backtest report

> Don't worry about this new tool just yet, we will explain in a future chapter how to interpret the following chart.

In [156]:
bt.plot(filename='backtest_report.html', open_browser=False)

/Users/neemaurassa/Library/Python/3.9/lib/python/site-packages/backtesting/_plotting.py:250: UserWarning: DatetimeFormatter scales now only accept a single format. Using the first provided: '%d %b'
  formatter=DatetimeTickFormatter(days=['%d %b', '%a %d'],
/Users/neemaurassa/Library/Python/3.9/lib/python/site-packages/backtesting/_plotting.py:250: UserWarning: DatetimeFormatter scales now only accept a single format. Using the first provided: '%m/%Y'
  formatter=DatetimeTickFormatter(days=['%d %b', '%a %d'],
/Users/neemaurassa/Library/Python/3.9/lib/python/site-packages/backtesting/_plotting.py:455: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df2 = (df.assign(_width=1).set_index('datetime')
/Users/neemaurassa/Library/Python/3.9/lib/python/site-packages/backtesting/_plotting.py:659: UserWarning: found multiple competing values for 'toolbar.active_drag' property; using the latest value
  fig = gridplot(
/Users/neemaurassa/Library/P

GridPlot(id='p1155', ...)

## How to invest based on the numerical increase?

> Instead of the direction (UP or DOWN)

Next chapter → [Backtesting with Regression Models]()

Classification Model | Regression Model
-|-
![](src/pred_classification.png) | ![](src/pred_regression.png)

Classification Strategy | Regression Strategy
-|-
![](src/res_classification.png) | ![](src/res_regression.png)